# UFLD on TuSimple: reproduce the ResNet-18 baseline (Colab)

Reproduction of *Ultra Fast Structure-aware Deep Lane Detection* (Qin et al., ECCV 2020). Code: <https://github.com/cfzd/Ultra-Fast-Lane-Detection>.

This notebook, top to bottom:
1. clones this repo and the original (pinned, for the golden tests), installs the **pinned** requirements;
2. downloads the full TuSimple dataset from Kaggle to Colab's **local disk** and converts it;
3. runs the **full test suite**, including golden tests against the original code on real frames;
4. evaluates the authors' pretrained checkpoint on **CUDA** and compares it with our MPS result;
5. **trains** ResNet-18 with `configs/tusimple_res18.py`, saving checkpoints and results to **Google Drive**;
6. evaluates the final checkpoint.

**Before you start**
- Runtime > Change runtime type > **T4 GPU**.
- Colab *Secrets* (key icon in the left sidebar), notebook access on:
  - `KAGGLE_USERNAME` and `KAGGLE_KEY` (from your `kaggle.json`), **or** `KAGGLE_API_TOKEN`.
  - `GITHUB_TOKEN` only if the repo is private (fine-grained token, *Contents: read*).
- The dataset is ~23 GB (unzipped ~23 GB more while extracting): this needs ~55 GB free on `/content`.

**If the session disconnects:** reconnect and *Run all* again. The dataset is re-downloaded to the new VM's local disk (it is not kept on Drive), but training **resumes from `last.pth` on Drive** at the next unfinished epoch (`scripts/train.py` auto-resumes when `--work-dir` already holds a `last.pth`). Resume is at epoch granularity, so at most one epoch of work is lost.

In [ ]:
# --- settings -------------------------------------------------------------------------------
GITHUB_REPO = "Ritesh-sudo/ufld-edge-lka"            # owner/name of this project's repo
REPO_COMMIT = None                                    # pin a commit/tag for a reproducible run; None = branch head
REFERENCE_COMMIT = "353df10"                          # original repo commit our golden tests were written against
RUN_NAME = "res18_tusimple"
DRIVE_DIR = "/content/drive/MyDrive/ufld"            # checkpoints, logs, results (survives disconnects)
DATA_DIR = "/content/tusimple"                       # local disk, NOT Drive
RAW_DIR = "/content/tusimple_raw"
NUM_WORKERS = 2                                       # free Colab VMs have 2 CPU cores

# Numbers to compare against
MPS_REFERENCE_ACC = 95.8154   # our port + authors' checkpoint, full TuSimple test set, Apple M4 (MPS)
README_ACC = 95.82            # original repo README, same checkpoint
PAPER_ACC = 95.87             # paper (arXiv:2004.11757v4), Table 5, Res18-Ours (accuracy only; no FP/FN reported)

In [ ]:
import os, shutil, subprocess, sys

def sh(cmd, check=True):
    """Run a shell command, stream its output, return it; raise if it fails (so Run-all stops)."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    if p.wait() and check:
        raise RuntimeError(f"command failed (exit {p.returncode}): {cmd}")
    return "".join(lines)

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
print("free disk on /content: %.0f GB" % (shutil.disk_usage("/content").free / 1e9))

In [ ]:
# Google Drive holds ONLY checkpoints, logs and results.
from google.colab import drive
drive.mount("/content/drive")
RUN_DIR = f"{DRIVE_DIR}/{RUN_NAME}"
RESULTS = f"{DRIVE_DIR}/results"
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
print("Drive run dir:", RUN_DIR, "| existing last.pth:", os.path.exists(f"{RUN_DIR}/last.pth"))

## 1. Code and environment

In [ ]:
REPO_DIR = "/content/ufld"
token = secret("GITHUB_TOKEN")
if not os.path.isdir(REPO_DIR):
    url = f"https://{token + '@' if token else ''}github.com/{GITHUB_REPO}.git"
    r = subprocess.run(["git", "clone", "-q", url, REPO_DIR], capture_output=True, text=True)
    if r.returncode:
        err = r.stderr.replace(token, "***") if token else r.stderr
        raise RuntimeError("git clone failed:\n" + err + "\nIf the repo is private, add a GITHUB_TOKEN Colab secret.")
    if token:  # do not leave the token in .git/config
        sh(f"git -C {REPO_DIR} remote set-url origin https://github.com/{GITHUB_REPO}.git")
else:
    sh(f"git -C {REPO_DIR} pull --ff-only")
if REPO_COMMIT:
    sh(f"git -C {REPO_DIR} checkout -q {REPO_COMMIT}")
os.chdir(REPO_DIR)

# The original repo: needed by the golden tests; pinned to the commit they were written against.
ref = "reference/Ultra-Fast-Lane-Detection"
if not os.path.isdir(ref):
    sh(f"git clone -q https://github.com/cfzd/Ultra-Fast-Lane-Detection {ref}")
    sh(f"git -C {ref} checkout -q {REFERENCE_COMMIT}")
sh("git log --oneline -1")

In [ ]:
# Pinned torch/torchvision first (same versions as the local MPS runs), then the rest.
sh("pip install -q torch==2.15.0 torchvision==0.29.2")
sh("pip install -q -r requirements.txt")

import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cuda", torch.version.cuda)
assert torch.__version__.split("+")[0] == "2.15.0" and torchvision.__version__.split("+")[0] == "0.29.2", \
    "Another torch was already loaded in this session: Runtime > Restart session, then Run all."
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
x = torch.randn(256, 256, device="cuda")
assert torch.isfinite(x @ x).all()
print("CUDA OK on", torch.cuda.get_device_name(0))

## 2. TuSimple dataset (Kaggle `manideep1108/tusimple`, ~23 GB) to local disk

The official TuSimple links are dead, so the Kaggle mirror is used. Kaggle serves train and test as separate folders;
`scripts/arrange_tusimple.py` merges them into the single `clips/` layout that `convert_tusimple.py` expects and fails
if any annotated image is missing. `convert_tusimple.py` then draws the lane masks and writes `train_gt.txt` / `test.txt`.

If Kaggle answers `429 Too Many Requests`, wait and rerun this cell; the download resumes.

In [ ]:
tok, user, key = secret("KAGGLE_API_TOKEN"), secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY")
if tok:
    os.environ["KAGGLE_API_TOKEN"] = tok
elif user and key:
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
elif not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    raise RuntimeError("Add Colab secrets KAGGLE_USERNAME + KAGGLE_KEY (or KAGGLE_API_TOKEN).")

if not os.path.exists(f"{DATA_DIR}/test.txt"):                 # test.txt is written last: its presence = done
    free_gb = shutil.disk_usage("/content").free / 1e9
    assert free_gb > 55, f"only {free_gb:.0f} GB free on /content; need ~55 GB"
    if not os.path.isdir(f"{RAW_DIR}/TUSimple"):
        sh(f"kaggle datasets download -d manideep1108/tusimple -p {RAW_DIR} --unzip")
    shutil.rmtree(DATA_DIR, ignore_errors=True)               # drop any half-finished earlier attempt
    sh(f"python scripts/arrange_tusimple.py --src {RAW_DIR} --dst {DATA_DIR}")
    shutil.rmtree(RAW_DIR, ignore_errors=True)
    sh(f"python scripts/convert_tusimple.py --root {DATA_DIR}")

os.makedirs("data", exist_ok=True)
if not os.path.exists("data/tusimple"):
    os.symlink(DATA_DIR, "data/tusimple")                     # the code and tests look in data/tusimple

n_train = sum(1 for _ in open(f"{DATA_DIR}/train_gt.txt"))
n_test = sum(1 for _ in open(f"{DATA_DIR}/test.txt"))
print(f"train images: {n_train} | test images: {n_test}")
assert n_test == 2782, f"expected 2782 test tasks, got {n_test}"
assert n_train == 3626, f"expected 3626 training images (label_data_0313/0531/0601), got {n_train}"

In [ ]:
# The authors' released TuSimple ResNet-18 checkpoint (from the original README).
os.makedirs("data/checkpoints", exist_ok=True)
CKPT = "data/checkpoints/tusimple_18.pth"
if not os.path.exists(CKPT):
    sh(f"gdown 1WCYyur5ZaWczH15ecmeDowrW30xcLrCn -O {CKPT}")
size_mb = os.path.getsize(CKPT) / 1e6
print(f"{CKPT}: {size_mb:.0f} MB")
assert 240 < size_mb < 250, "unexpected checkpoint size; Drive may have returned an error page"

## 3. Test suite (incl. golden tests vs the original code on real frames)

`UFLD_GOLDEN_N` is the number of evenly spaced real training images the real-data golden tests compare
(converter masks, row-anchor labels, image tensors, seeded augmentation). Nothing may be skipped here.

In [ ]:
os.environ["UFLD_GOLDEN_N"] = "200"
out = sh("python -m pytest -q -ra -p no:cacheprovider")
assert " skipped" not in out, "some tests were skipped: data, checkpoint or reference repo is missing (see -ra lines above)"

## 4. Authors' pretrained checkpoint on CUDA vs our MPS result

In [ ]:
sh(f"python scripts/eval_tusimple.py --config configs/tusimple_res18.py --ckpt {CKPT} "
   f"--device cuda --set num_workers={NUM_WORKERS} --out {RESULTS}/pretrained_eval_cuda.csv")

import csv
row = list(csv.DictReader(open(f"{RESULTS}/pretrained_eval_cuda.csv")))[-1]
cuda_acc = float(row["accuracy_pct"])
print(f"authors' checkpoint | CUDA {cuda_acc:.4f}%  | MPS {MPS_REFERENCE_ACC:.4f}%  | diff {cuda_acc - MPS_REFERENCE_ACC:+.4f} pts")
print(f"                    | README {README_ACC}%  | paper (Table 5) {PAPER_ACC}%  | FP {row['fp']}  FN {row['fn']}")
assert abs(cuda_acc - MPS_REFERENCE_ACC) < 0.1, \
    "CUDA and MPS disagree by more than 0.1 pt: investigate before training (data layout? torch version? TF32?)"

## 5. Train ResNet-18 (`configs/tusimple_res18.py`: 100 epochs, batch 32, Adam 4e-4, cosine, aux branch on)

Checkpoints go to Drive: `last.pth` every epoch (written atomically), `epNNN.pth` every 10 epochs
(each ~0.75 GB with optimizer state, so budget ~8 GB of Drive). Re-running this cell after a disconnect resumes from `last.pth`.

In [ ]:
sh(f"python scripts/train.py --config configs/tusimple_res18.py --work-dir {RUN_DIR} --set num_workers={NUM_WORKERS}")

In [ ]:
# Optional: training curves
%load_ext tensorboard
%tensorboard --logdir {RUN_DIR}

## 6. Evaluate the trained model

In [ ]:
sh(f"python scripts/eval_tusimple.py --config configs/tusimple_res18.py --ckpt {RUN_DIR}/last.pth "
   f"--device cuda --set num_workers={NUM_WORKERS} --out {RESULTS}/{RUN_NAME}_eval.csv")

mine = list(csv.DictReader(open(f"{RESULTS}/{RUN_NAME}_eval.csv")))[-1]
acc = float(mine["accuracy_pct"])
print("| model                        | Accuracy | FP     | FN     |")
print("|------------------------------|----------|--------|--------|")
print(f"| authors' ckpt (our port, CUDA) | {cuda_acc:7.2f}% | {float(row['fp']):.4f} | {float(row['fn']):.4f} |")
print(f"| trained here                 | {acc:7.2f}% | {float(mine['fp']):.4f} | {float(mine['fn']):.4f} |")
print(f"| paper, Table 5 (Res18-Ours)  | {PAPER_ACC:7.2f}% |  n/a   |  n/a   |   <- paper reports accuracy and runtime only")
print(f"\ngap to paper: {acc - PAPER_ACC:+.2f} pt | gap to authors' released checkpoint: {acc - README_ACC:+.2f} pt")
print("Results are in", RESULTS)

## Notes for the write-up
- Report the gap to the paper **and** to the released checkpoint honestly, with likely causes (epochs, batch size, augmentation, seed, framework version); one seed is one sample.
- Copy `results/*.csv` from Drive into the repo's `results/` folder and commit them.
- FP/FN are the official TuSimple metrics and are reported for our runs; the paper's Table 5 has accuracy only.